In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.mixture import GaussianMixture
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import FunctionTransformer


In [2]:
dataset = 'news_reducido.csv'

# Leer los datos en formato csv
data = pd.read_csv(dataset, on_bad_lines='skip')
# Nos quedamos con el texto (puedes quedarte con más información si quieres)
X = data['text'].fillna('').astype(str).to_numpy() # Ponco con un espacio los espacio nulos

In [8]:
# Ahora, procesamos las etiquetas, para cada clase, le asignamos un valor numérico entre 0 y el número de clases
semilla1=42
semilla2=640
semilla3=5300742

semillas = [42, 640, 5300742]
pesos = ["uniform", "distance"]
valor_p = [1, 2, 3, 5, 7, 10]
valor_k = [3, 4, 5, 6, 7, 10]


semilla = semilla1

enc = OrdinalEncoder()
y = enc.fit_transform(np.reshape(data['category'], (-1,1))).reshape(-1)

# Hacemos la partición train-test con Validacion cruzada estratificada
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=semilla)
skf.get_n_splits(X, y)

# Definir aquí los pipelines necesarios para cada problema (clustering, clasificación, etc.)
text_clasifier_binario = Pipeline([
    ('vect', CountVectorizer(binary=True)),
    ('clf', KNeighborsClassifier()) #Preguntar los randoms state
])

text_clasifier_tf = Pipeline([
    ('vect', CountVectorizer(binary=False)),
    ('clf', KNeighborsClassifier()) #Preguntar el random_state
])

text_clasifier_tfidf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', KNeighborsClassifier())
])

text_sgd = text_clasifier_tfidf


In [9]:
accuracies = np.zeros(5)
for i, (tra, tst) in enumerate(skf.split(X,y)):
    fit_classification = True
    # Clasificacion
    if fit_classification:
        # Entrenamientoo
        text_sgd.fit(X[tra], y[tra])

        # Test (obtener predicciones)
        predicted = text_sgd.predict(X[tst])

        # Calculo de metricas de calidad (ahora, solo accuracy)
        acc = np.mean(predicted == y[tst])

        print(acc)
        accuracies[i] = acc
# Tras el K-Fold, hay que mostrar la precision media obtenida ( o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

0.3595
0.3365
0.3735
0.3645
0.3505
Precision media = 0.3569
